# Working with datetimes: parse, extract, zones, resample

_Data Wrangling_

**Turn messy date strings into real timestamps you can sort, slice, and aggregate by time.**

A datetime is not a string and not just a number &mdash; it is a point on a timeline with a
       calendar (years, months, leap days) and, optionally, a time zone. Pandas gives you a dedicated
       type, datetime64[ns], plus a Timestamp for a single instant and a
       Timedelta for a duration. Once a column is this type, four powerful things open up.

       (1) Parse. pd.to_datetime reads varied string formats into real timestamps.
       (2) Extract. The .dt accessor pulls out calendar pieces &mdash; year, month, day,
       hour, and dayofweek (Monday = 0 ... Sunday = 6). (3) Localize / convert.
       tz_localize stamps a naive time with a zone; tz_convert moves an aware time
       to another zone. (4) Index and aggregate. Put the timestamps on the index (a
       DatetimeIndex) and you can slice by date range, .resample() to a coarser cadence,
       and take .rolling() windows.

---

This notebook builds the datetime workflow one piece at a time. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — pandas + numpy

### Step 1 — Parse messy date strings into real timestamps

`pd.to_datetime` reads varied string formats into proper `Timestamp` values. Passing `dayfirst=True` tells it the strings are `DD/MM/YYYY`, and `errors="coerce"` turns anything unparseable into `NaT` instead of raising. An explicit `format="%d/%m/%Y"` is faster still and rejects anything that does not match that exact shape.

In [ ]:
import numpy as np
import pandas as pd

# A column of varied / ambiguous date strings, including one piece of junk.
raw = pd.Series(["01/03/2023", "15/03/2023", "31/03/2023",
                 "10/04/2023", "not a date"])

# dayfirst=True -> DD/MM/YYYY; errors='coerce' turns junk into NaT.
parsed = pd.to_datetime(raw, dayfirst=True, errors="coerce")

# An explicit format is faster and rejects anything that doesn't match:
parsed = pd.to_datetime(raw, format="%d/%m/%Y", errors="coerce")

print(parsed.tolist())
# [Timestamp('2023-03-01'), ..., Timestamp('2023-04-10'), NaT]

### Step 2 — Extract calendar pieces with the `.dt` accessor

Once a column holds real timestamps, the `.dt` accessor pulls out calendar fields one at a time: `year`, `month`, `day`, and `dayofweek`. Note that `dayofweek` is zero-indexed with Monday = 0 and Sunday = 6. We build a small frame so each extracted component sits in its own column next to the timestamp it came from.

In [ ]:
# Put the parsed timestamps in a frame, then add one column per component.
comp = pd.DataFrame({"ts": parsed})
comp["year"]      = comp["ts"].dt.year
comp["month"]     = comp["ts"].dt.month
comp["day"]       = comp["ts"].dt.day
comp["dayofweek"] = comp["ts"].dt.dayofweek      # Mon=0 ... Sun=6

print(comp)

### Step 3 — Localize a time, convert it, and do date arithmetic

A naive timestamp is just a wall-clock reading with no zone. `tz_localize` stamps it with the zone it was recorded in; `tz_convert` then re-expresses that same instant in another zone (here UTC). Subtracting two timestamps yields a `Timedelta` duration, and adding a `Timedelta` to a timestamp shifts it forward — useful for deadlines.

In [ ]:
# --- Time zones: localize at the edge, then work in UTC ---
t = pd.Timestamp("2023-03-01 09:30:00")          # naive wall-clock reading
t_ny  = t.tz_localize("America/New_York")        # attach the source zone
t_utc = t_ny.tz_convert("UTC")                   # same instant, expressed in UTC
print(t_ny, "->", t_utc)                         # ...-05:00 -> 14:30:00+00:00

# --- Date arithmetic / durations (Timedelta) ---
span = pd.Timestamp("2023-04-10") - pd.Timestamp("2023-03-01")
print(span, span.days)                           # 40 days 00:00:00   40

deadline = pd.Timestamp("2023-03-01") + pd.Timedelta(days=14)
print(deadline)                                  # 2023-03-15 00:00:00

### Step 4 — Index by time, then resample and take a rolling window

Put timestamps on the index (a `DatetimeIndex`) and time-aware aggregation opens up. We synthesize a year of daily values with a trend, a yearly season, a weekday bump, and noise. `resample("M").mean()` buckets the daily series into monthly means, and `rolling(7).mean()` takes a 7-day backward moving average that smooths the day-to-day jitter.

In [ ]:
# Build a year of synthetic daily data on a DatetimeIndex.
rng_ = np.random.RandomState(7)
idx = pd.date_range("2023-01-01", periods=365, freq="D")   # a DatetimeIndex
t_ = np.arange(365)
daily = (100 + 0.15 * t_                                    # upward trend
         + 20 * np.sin(2 * np.pi * t_ / 365)               # yearly season
         + 5 * (idx.dayofweek < 5)                         # weekday bump
         + rng_.normal(0, 4, 365))                         # noise
s = pd.Series(daily, index=idx)

monthly = s.resample("M").mean()        # bucket daily -> monthly means
roll7   = s.rolling(7).mean()           # 7-day backward moving average

print(monthly.round(1).head())
print(roll7.dropna().round(1).head())

## Visualize the data & results

_On a real-ish noisy daily series, what does a 7-day rolling mean do that the raw daily values do not?_

### Step 5 — Rebuild the daily series for plotting

To compare smoothing methods we recreate the same noisy daily series (same seed, so the numbers match Step 4). Each component — trend, yearly seasonality, weekday bump, and noise — is added explicitly so you can see what goes into the signal before we smooth it.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.RandomState(7)
idx = pd.date_range("2023-01-01", periods=365, freq="D")   # DatetimeIndex
t = np.arange(365)
daily = (100 + 0.15 * t                      # upward trend
         + 20 * np.sin(2 * np.pi * t / 365)  # yearly seasonality
         + 5 * (idx.dayofweek < 5)           # weekday bump via .dt-style dayofweek
         + rng.normal(0, 4, 365))            # daily noise
s = pd.Series(daily, index=idx)

### Step 6 — Smooth with a rolling mean and coarsen with resample

A 7-day rolling mean replaces each day with the average of it and the prior six days, so the weekly wobble and noise flatten out while the trend stays. We print a 60-day window (starting at index 6, where the rolling average first becomes defined) to compare raw vs smoothed. Resampling to monthly means gives an even coarser, trend-only view.

In [ ]:
roll7 = s.rolling(7).mean()                  # 7-day backward moving average
sub = slice(6, 66)                           # 60 plotted days, rolling defined
print("raw :", np.round(s.values[sub], 1))
print("roll:", np.round(roll7.values[sub], 1))

monthly = s.resample("M").mean()             # bucket -> monthly means
print("monthly:", np.round(monthly.values, 1))
# monthly -> [110.5 123.9 134.  139.  136.9 134.4 127.7 124.4 123.5 127.1 136.8 149.9]

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** A column of order dates looks like ["03/04/2023", "05/06/2023", "11/12/2023"] and your European colleague says all three are day-first (DD/MM/YYYY). You run pd.to_datetime(col) with no extra arguments. What can go wrong and how do you fix it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Notice every value is ambiguous: each part is &le; 12, so pandas cannot tell day from month. — _"03/04/2023" parses as March 4th (MM/DD) by pandas' default, but the intended value is April 3rd (DD/MM)._
- Recognize the failure is silent &mdash; no error, just wrong dates. — _Silent corruption is the worst kind: downstream months, weekdays, and sorts are all off, but nothing warns you._
- Re-parse with the day-first convention. — _pd.to_datetime(col, dayfirst=True) (or an explicit format="%d/%m/%Y") pins the order so the parse matches the source._

**Answer:** With no arguments pandas uses month-first, so April 3rd silently becomes March 4th and every interpretation downstream is wrong. Fix it by stating the order: pd.to_datetime(col, dayfirst=True), or best, an explicit format="%d/%m/%Y", which is faster and refuses anything that does not match.

</details>

**Problem 2.** You have event timestamps from users in New York and London, all stored as naive wall-clock strings. You want correct counts of events per UTC day. Outline the steps.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Parse the strings to datetimes, then localize each to the user's own zone. — _A naive timestamp has no zone; tz_localize("America/New_York") / tz_localize("Europe/London") says which wall clock the reading came from._
- Convert every localized timestamp to UTC with tz_convert("UTC"). — _Now all events share one reference instant, so 23:30 New York and 23:30 London no longer collide on the wrong day._
- Set the UTC column as the index and .resample("D").count(). — _Resampling needs a DatetimeIndex; counting per UTC day gives consistent buckets across both regions._

**Answer:** Localize, then convert, then resample. Localize each naive timestamp to its source zone (tz_localize), convert all to UTC (tz_convert("UTC")) so every event agrees on the instant, then set the UTC column as the index and .resample("D").count(). Doing the math in UTC sidesteps Daylight Saving Time gaps and the naive-vs-aware mixing error.

</details>

**Problem 3.** A daily sales series is very noisy and you also need an honest train/test split for a forecast. What two datetime techniques apply, and what is the one rule the split must obey?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Smooth the noise with a backward rolling mean: s.rolling(7).mean(). — _A 7-day moving average averages each day with the prior six, revealing the trend and weekly shape without using any future points._
- Optionally resample to a coarser cadence for a higher-level view: s.resample("M").mean(). — _Monthly means strip out day-to-day jitter entirely when you only care about the slow trend._
- Split on a cutoff date, not at random. — _Train strictly on dates before the cutoff and test on dates after it, so the model never sees the future during training._

**Answer:** Use a rolling mean (.rolling(7).mean()) to smooth and, optionally, resampling (.resample("M").mean()) for a coarser view. The split rule: cut on a date, train before it, test after it &mdash; a random split leaks the future and inflates your measured accuracy. Both the rolling mean and the split look only backward, which is what keeps them honest.

</details>